# GeCo2 AERO EYES finetune -- chạy trên vast.ai (nhánh `test-geco2`)

Notebook này: cài dependency (giống hệt `notebooks/aero_eyes_vastai_geco2.ipynb`) -> clone code +
GECO2 + checkpoint + dataset -> **finetune GeCo2 trên 14 video training** (loại bỏ shape token,
domain-randomization qua downscale ref ảnh) -> đánh giá checkpoint đã finetune trên 6 video test
held-out bằng pipeline `aero_eyes.stages.run_all`/`aero_eyes.evaluate` KHÔNG SỬA GÌ.

Xem `docs/GECO2_FINETUNE_PLAN.md` để biết đầy đủ lý do thiết kế.

**Notebook này KHÔNG sửa `notebooks/aero_eyes_vastai_geco2.ipynb`** -- đây là 1 notebook mới,
độc lập, tái sử dụng đúng chuỗi cài đặt môi trường đã kiểm chứng ở notebook đó (mục 1-9 dưới đây
gần như giống hệt, xem notebook kia để biết chi tiết từng bước / cách xử lý lỗi thường gặp).

**Lưu ý riêng cho vast.ai:**
- Không có `google.colab.drive.mount()` hay `/kaggle/input` tự động -- dataset phải tự tải bằng `gdown`.
- Chọn template vast.ai có sẵn **CUDA toolkit đầy đủ** (không chỉ runtime) -- bước build extension
  CUDA (mục 5) cần `nvcc`. Image kiểu `pytorch/pytorch:*-devel` hoặc template ghi "CUDA" là an toàn nhất.
- `models/counter.py::CNT` (biến thể TRAINING, có aux head) cần đúng extension CUDA
  `MultiScaleDeformableAttention` như biến thể inference-only (`counter_infer.py::CNT`) --
  KHÔNG có fallback CPU, bước build ở mục 5 là bắt buộc cho cả training lẫn inference.
- Instance vast.ai có thể bị xoá bất cứ lúc nào sau khi dừng thuê -- **tải checkpoint + kết quả về
  trước khi kết thúc** (cell cuối cùng có hướng dẫn).

## 0. Biến cấu hình chung -- chỉnh ở đây trước khi chạy

In [ ]:
import os

# --- Repo chính (branch test-geco2) ---
REPO_URL = "https://github.com/Hoang-hai-yen/Test.git"
REPO_BRANCH = "test-geco2"
REPO_DIR = "/workspace/Test"

# --- GECO2 upstream (bu cho gitlink rong -- xem GECO2_FINETUNE_PLAN.md) ---
GECO2_UPSTREAM_URL = "https://github.com/jerpelhan/GECO2.git"
GECO2_PINNED_COMMIT = "5b4fe9c4bc4a453bb366d314b80efb81450c51ef"

# --- GeCo2 pretrained (base) weights -- the STARTING POINT for finetuning, never overwritten ---
GECO2_WEIGHTS_URL = "https://huggingface.co/datasets/jerpelhan/geco2-assets/resolve/main/weights/CNTQG_multitrain_ca44.pth?download=true"

# --- Dataset tren Google Drive: 1 file .zip chua CA 20 video (14 train + 6 test) ---
# Lay FILE_ID tu link share dang https://drive.google.com/file/d/<FILE_ID>/view
GDRIVE_FILE_ID = "PASTE_HERE"  # <-- ACTION REQUIRED: dan File ID that vao day

os.environ.update({
    "REPO_URL": REPO_URL, "REPO_BRANCH": REPO_BRANCH, "REPO_DIR": REPO_DIR,
    "GECO2_UPSTREAM_URL": GECO2_UPSTREAM_URL, "GECO2_PINNED_COMMIT": GECO2_PINNED_COMMIT,
    "GECO2_WEIGHTS_URL": GECO2_WEIGHTS_URL, "GDRIVE_FILE_ID": GDRIVE_FILE_ID,
})
print("OK -- nho dien GDRIVE_FILE_ID that truoc khi chay cell tai dataset (muc 8).")

## 1. Kiem tra GPU / CUDA toolkit

In [ ]:
!nvidia-smi
!echo "--- nvcc (can cho buoc build CUDA extension o duoi) ---"
!nvcc --version || echo "CANH BAO: khong thay nvcc -- chon lai template vast.ai co CUDA toolkit (devel), khong phai ban runtime-only." 

## 1a. Dong bo python giua Jupyter kernel va shell (`%%bash`)

Xem `notebooks/aero_eyes_vastai_geco2.ipynb` muc "1a" cho ly do day du. Chay cell nay SOM NHAT.

In [ ]:
import os
import shutil
import sys

kernel_python = sys.executable
kernel_bin = os.path.dirname(kernel_python)
shell_python3 = shutil.which("python3")

print("Kernel python (sys.executable):", kernel_python)
print("Shell python3 (which python3) truoc khi sua:", shell_python3)

if shell_python3 and os.path.realpath(shell_python3) != os.path.realpath(kernel_python):
    print(">>> LECH MOI TRUONG -- uu tien thu muc cua kernel python len dau PATH.")
else:
    print(">>> Khop nhau (hoac khong xac dinh duoc shell python3) -- van set PATH cho chac.")

os.environ["PATH"] = kernel_bin + os.pathsep + os.environ.get("PATH", "")

print("Shell python3 sau khi sua:", shutil.which("python3"))
print("pip sau khi sua:", shutil.which("pip"))

## 1b. Dam bao lenh `python` ton tai

In [ ]:
%%bash
set -e
if ! command -v python &> /dev/null; then
    PY3=$(command -v python3)
    echo "Khong co lenh 'python' -- tao symlink toi $PY3"
    ln -sf "$PY3" /usr/local/bin/python
fi
python --version
python -m pip --version

## 2. Lay code (clone lan dau, pull cac lan sau)

An toan de chay lai nhieu lan: neu `$REPO_DIR` da ton tai, cell nay `git pull` thay vi xoa-clone-lai --
giu nguyen `GECO2/models/ops/` (da build o muc 5) va `GECO2/CNTQG_multitrain_ca44.pth` (da tai o muc 7).

In [ ]:
%%bash
set -e
mkdir -p /workspace

if [ -d "$REPO_DIR/.git" ]; then
    echo ">>> $REPO_DIR da ton tai -- pull code moi (KHONG xoa GECO2/models/ops hay weights da tai)."
    cd "$REPO_DIR"
    git fetch origin "$REPO_BRANCH"
    git checkout "$REPO_BRANCH"
    git reset --hard "origin/$REPO_BRANCH"
else
    echo ">>> Chua co $REPO_DIR -- clone moi."
    git clone --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_DIR"
    cd "$REPO_DIR"
fi

if [ -z "$(ls -A GECO2 2>/dev/null)" ]; then
    echo ">>> GECO2/ rong (gitlink chua dang ky) -- clone bu truc tiep tu upstream ..."
    rm -rf GECO2
    git clone "$GECO2_UPSTREAM_URL" GECO2
    cd GECO2 && git checkout "$GECO2_PINNED_COMMIT" && cd ..
else
    echo ">>> GECO2/ da co san code -- giu nguyen, khong dung vao."
fi

echo "--- kiem tra ---"
ls "$REPO_DIR"
ls "$REPO_DIR/GECO2" | head -5

echo "--- trang thai build/weights trong GECO2/ (khong bi anh huong boi git pull) ---"
[ -d "$REPO_DIR/GECO2/models/ops" ] && echo "models/ops: da build (muc 5 khong can chay lai)" \
    || echo "models/ops: CHUA build -- can chay muc 5"
[ -f "$REPO_DIR/GECO2/CNTQG_multitrain_ca44.pth" ] && echo "weights: da co (muc 7 khong can chay lai)" \
    || echo "weights: CHUA tai -- can chay muc 7" 

## 3. Cai dependency cua aero_eyes

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
pip install -q -r requirements.txt
pip install -q -e .
pip install -q opencv-contrib-python-headless
echo "Done: aero_eyes deps" 

## 4. Cai dependency rieng cua GECO2

Xem `notebooks/aero_eyes_vastai_geco2.ipynb` muc 4 cho ly do day du cua tung dong lenh duoi day
(`--no-deps` cho mobile_sam, bo `huggingface-hub` pin, v.v.).

In [ ]:
%%bash
set -e
pip install -q hydra-core omegaconf iopath scikit-image pycocotools einops
pip install -q timm
pip install -q --no-deps git+https://github.com/ChaoningZhang/MobileSAM.git
echo "--- kiem tra torch/torchvision con khop nhau khong ---"
python -c "import torch, torchvision; print('torch', torch.__version__, '| torchvision', torchvision.__version__); from torchvision.ops import nms; import torch as t; nms(t.tensor([[0.,0.,1.,1.]]), t.tensor([0.9]), 0.5); print('torchvision::nms OK')"
echo "Done: GECO2 deps (chua pin numpy/pydantic -- xem cell pin version o cuoi)" 

### (Khac phuc) torchvision loi `has no attribute 'extension'` / `operator torchvision::nms does not exist`

Chi chay neu muc 4 van bao loi tren dua co `--no-deps`. Xem `aero_eyes_vastai_geco2.ipynb` muc tuong ung.

In [ ]:
%%bash
set -e
pip install -q --force-reinstall --no-deps \
    torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 \
    --index-url https://download.pytorch.org/whl/cu126
echo "--- kiem tra ---"
python -c "
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda available:', torch.cuda.is_available())
from torchvision.ops import nms
nms(torch.tensor([[0.,0.,1.,1.]]), torch.tensor([0.9]), 0.5)
print('torchvision::nms OK')
" 

### (Tuy chon) Fix `nvcc fatal: Unsupported gpu architecture 'compute_70'`

Chi chay neu buoc build CUDA extension (muc 5) bao loi dung nhu tren (GPU Volta/V100 cu, nvcc he
thong qua moi da bo ho tro compute_70). Xem `aero_eyes_vastai_geco2.ipynb` muc tuong ung cho giai
thich day du.

In [ ]:
# Dat True neu build CUDA extension (muc 5) bao loi "nvcc fatal: Unsupported gpu architecture 'compute_70'".
FIX_NVCC_FOR_VOLTA = False

import os
import torch
print("torch hien tai:", torch.__version__, "| cuda build:", torch.version.cuda,
      "| cuda available:", torch.cuda.is_available())

os.environ["FIX_NVCC_FOR_VOLTA"] = "1" if FIX_NVCC_FOR_VOLTA else "0" 

In [ ]:
%%bash
set -e
if [ "$FIX_NVCC_FOR_VOLTA" != "1" ]; then
    echo "FIX_NVCC_FOR_VOLTA=False -- bo qua cell nay."
    exit 0
fi

. /etc/os-release
echo "OS phat hien: $ID $VERSION_ID"
TAG="${ID}$(echo "$VERSION_ID" | tr -d '.')"
echo "Thu NVIDIA apt repo tag: $TAG"

KEYRING_URL="https://developer.download.nvidia.com/compute/cuda/repos/${TAG}/x86_64/cuda-keyring_1.1-1_all.deb"
if ! wget -q --spider "$KEYRING_URL"; then
    echo "Khong co repo cho '$TAG', thu fallback 'ubuntu2404' ..."
    TAG="ubuntu2404"
    KEYRING_URL="https://developer.download.nvidia.com/compute/cuda/repos/${TAG}/x86_64/cuda-keyring_1.1-1_all.deb"
fi
echo "Dung: $KEYRING_URL"

wget -q "$KEYRING_URL" -O /tmp/cuda-keyring.deb
dpkg -i /tmp/cuda-keyring.deb
apt-get update -qq
apt-get install -y -qq cuda-toolkit-12-6

echo "--- kiem tra ---"
ls -d /usr/local/cuda-12.6
/usr/local/cuda-12.6/bin/nvcc --version

In [ ]:
if FIX_NVCC_FOR_VOLTA:
    import os, subprocess, sys

    cuda126_home = "/usr/local/cuda-12.6"
    nvcc126 = os.path.join(cuda126_home, "bin", "nvcc")
    if not os.path.exists(nvcc126):
        raise FileNotFoundError(
            f"{nvcc126} khong ton tai -- cell apt cai CUDA toolkit o tren co the da loi, "
            "doc lai log cell do truoc khi chay tiep."
        )

    os.environ["CUDA_HOME"] = cuda126_home
    os.environ["PATH"] = os.path.join(cuda126_home, "bin") + os.pathsep + os.environ["PATH"]

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore",
                     "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
                     "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)

    print("--- kiem tra lai ---")
    print("CUDA_HOME =", os.environ["CUDA_HOME"])
    subprocess.run(["nvcc", "--version"], check=True)

## 5. Build CUDA extension (Deformable-DETR ops) -- bat buoc cho ca training lan inference

`GECO2/models/query_generator.py::C_base` (dung boi CA `models/counter.py::CNT` -- bien the
TRAINING dung o notebook nay -- LAN `models/counter_infer.py::CNT`) can
`models.ops.modules.ms_deform_attn.MSDeformAttn`. Buoc nay build extension CUDA roi copy source
wrapper vao dung vi tri `GECO2/models/ops/` (giong het `GECO2/install.sh`).

In [ ]:
%%bash
set -e
cd "$REPO_DIR/GECO2/Deformable-DETR/models/ops"
CUDA_VISIBLE_DEVICES=0 python -m pip install --no-build-isolation .
cd "$REPO_DIR/GECO2"
rm -rf ./models/ops
cp -r ./Deformable-DETR/models/ops ./models/ops
python -c "import torch; import MultiScaleDeformableAttention; print('MultiScaleDeformableAttention import OK')"
echo "Done: CUDA ops extension built + copied" 

## 6. Pin version cuoi cung (chay SAU CUNG trong phan cai dat)

`scipy<1.13` them vao day vi ly do CUNG NHU numpy/pydantic ben duoi: cac lenh pip install o tren
(hydra-core, mobile_sam, ...) co the keo theo 1 ban scipy moi (>=1.13 tro len chi ho tro
numpy>=2.0) MA KHONG BIET numpy sap bi ep xuong <2 o chinh cell nay -- neu khong pin lai, import
`scipy.optimize.linear_sum_assignment` trong `GECO2/models/matcher.py` (dung boi
`scripts/train_geco2_aeroeyes.py`) se bao loi `AttributeError: module 'numpy' has no attribute
'long'` (scipy ban moi dung `np.long`/`np.ulong`, cac alias nay khong ton tai truoc numpy 2.0).
Loi nay KHONG xay ra khi chi chay pipeline INFERENCE (`aero_eyes_vastai_geco2.ipynb`) vi
`GECO2/models/matcher.py` chi duoc dung cho training, khong duoc dung boi
`aero_eyes/models/geco2_detector.py`.

In [ ]:
%%bash
set -e
pip install -q "numpy<2"
pip install -q --force-reinstall "pydantic<2.11"
pip install -q "scipy<1.13"
python -c "import numpy, pydantic, scipy; print('numpy', numpy.__version__, '| pydantic', pydantic.VERSION, '| scipy', scipy.__version__)" 

## 7. Tai trong so GeCo2 GOC (base checkpoint -- diem xuat phat de finetune, khong bao gio bi ghi de)

In [ ]:
%%bash
set -e
cd "$REPO_DIR/GECO2"
wget -q --show-progress -O CNTQG_multitrain_ca44.pth "$GECO2_WEIGHTS_URL"
ls -lh CNTQG_multitrain_ca44.pth

## 8. Tai dataset tu Google Drive (.zip) -- can CA 20 video (14 training + 6 test)

**ACTION REQUIRED:** dien `GDRIVE_FILE_ID` that o Cell 0 truoc khi chay cell nay.

In [ ]:
%%bash
set -e
if [ "$GDRIVE_FILE_ID" = "PASTE_HERE" ]; then
    echo "CHUA dien GDRIVE_FILE_ID that o Cell 0 -- bo qua buoc tai dataset."
    echo "Cac cell setup o tren van dung duoc de test rieng; quay lai day khi co File ID."
else
    pip install -q -U gdown
    mkdir -p /workspace/aero_eyes_dataset_raw
    gdown "$GDRIVE_FILE_ID" -O /workspace/aero_eyes_dataset.zip
    unzip -q -o /workspace/aero_eyes_dataset.zip -d /workspace/aero_eyes_dataset_raw
    echo "--- cau truc sau khi giai nen (kiem tra ky truoc khi set DATA_ROOT o duoi) ---"
    find /workspace/aero_eyes_dataset_raw -maxdepth 4
fi

## 9. Tro dung duong dan dataset

**Xem output `find` o cell tren roi chinh `DATA_ROOT` cho khop** -- Drive/zip co the co them 1 lop
thu muc con tuy cach ban nen, dung doan.

In [ ]:
import os

# <-- CHINH theo output `find` o cell tren
DATA_ROOT = "/workspace/aero_eyes_dataset_raw"
WORK_DIR = "/workspace/runs/geco2_finetune"

os.environ.update({"DATA_ROOT": DATA_ROOT, "WORK_DIR": WORK_DIR})

samples = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))] if os.path.isdir(DATA_ROOT) else []
print(f"Tim thay {len(samples)} sample(s):", sorted(samples))

## 10. Cau hinh finetune -- 2 file GT KHAC NHAU, dung nham la loi de gap nhat

`data.gt.global_file` mac dinh trong `configs/config.yaml` la file GT 6-VIDEO TEST
(`annotations (1).json`) -- **KHONG dung file nay de training**. `TRAIN_GT_FILE` duoi day tro
rieng vao file GT 14-video training (`annotations.json`). `scripts/train_geco2_aeroeyes.py` cung
tu assert(raise) neu vo tinh dua nham 6 video test vao training, nhung dat 2 bien ro rang o day
giup tranh nham lan tu dau.

In [ ]:
import os

TRAIN_GT_FILE = f"{DATA_ROOT}/annotations.json"        # 14 video TRAINING
TEST_GT_FILE = f"{DATA_ROOT}/annotations (1).json"      # 6 video TEST (held-out, danh gia o muc 14)

HOLDOUT_CATEGORIES = ["Lifering", "Person1"]  # internal train/val split -- xem GECO2_FINETUNE_PLAN.md muc 6

EPOCHS = 15
STEPS_PER_EPOCH = 400
BATCH_SIZE = 1
LR = 1e-4
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 0.1
AUX_WEIGHT = 0.3
P_PRESENT = 0.5
REF_DOWNSCALE_LO = 0.03
REF_DOWNSCALE_HI = 1.0
EARLY_STOP_PATIENCE = 5

BASE_CHECKPOINT = f"{REPO_DIR}/GECO2/CNTQG_multitrain_ca44.pth"
OUT_CHECKPOINT = f"{REPO_DIR}/GECO2/CNTQG_aeroeyes_finetuned.pth"

# IMPORTANT: %%bash cells run in a SEPARATE shell subprocess that only sees
# explicitly-exported environment variables -- a plain Python variable like
# `EPOCHS = 15` above is invisible to `$EPOCHS` in a later %%bash cell.
# Every value referenced by muc 12/13's `%%bash` cells must be exported here.
os.environ.update({
    "TRAIN_GT_FILE": TRAIN_GT_FILE, "TEST_GT_FILE": TEST_GT_FILE,
    "BASE_CHECKPOINT": BASE_CHECKPOINT, "OUT_CHECKPOINT": OUT_CHECKPOINT,
    "HOLDOUT_CATEGORIES": " ".join(HOLDOUT_CATEGORIES),  # space-separated -> bash word-splits into argparse nargs="+"
    "EPOCHS": str(EPOCHS), "STEPS_PER_EPOCH": str(STEPS_PER_EPOCH), "BATCH_SIZE": str(BATCH_SIZE),
    "LR": str(LR), "WEIGHT_DECAY": str(WEIGHT_DECAY), "MAX_GRAD_NORM": str(MAX_GRAD_NORM),
    "AUX_WEIGHT": str(AUX_WEIGHT), "P_PRESENT": str(P_PRESENT),
    "REF_DOWNSCALE_LO": str(REF_DOWNSCALE_LO), "REF_DOWNSCALE_HI": str(REF_DOWNSCALE_HI),
    "EARLY_STOP_PATIENCE": str(EARLY_STOP_PATIENCE),
})
print("TRAIN_GT_FILE:", TRAIN_GT_FILE, "| exists:", os.path.exists(TRAIN_GT_FILE))
print("TEST_GT_FILE :", TEST_GT_FILE, "| exists:", os.path.exists(TEST_GT_FILE))

## 11. Kiem tra chuyen doi toa do GT -> canvas (BAT BUOC truoc khi training)

`convert_gt_box_to_canvas` la ham DUY NHAT chuyen GT box tu pixel-goc-cua-video sang canvas da
resize/pad -- dung boi CA training loop LAN cell nay, nen "cai minh nhin thay" va "cai model duoc
train tren" khong bao gio lech nhau. Mot loi lech toa do (vd sai huong padding) van co the tao ra
loss curve giam "hop ly" -- **KHONG duoc bo qua buoc nay**, phai xac nhan bang mat truoc khi tin
tuong bat ky training run nao.

In [ ]:
import os
os.chdir(REPO_DIR)  # duong dan tuong doi trong config.yaml (./GECO2, configs/config.yaml, ...)

import cv2
import numpy as np

from aero_eyes.config import load_config
from aero_eyes.models.geco2_detector import GeCo2Detector
from aero_eyes.models.geco2_finetune_data import (
    EXPECTED_TRAIN_VIDEOS, convert_gt_box_to_canvas, split_train_val,
)
from aero_eyes.stages.stage123_geco2 import _locate_video
from aero_eyes.utils.io import list_video_ids, load_gt
from aero_eyes.utils.video import read_frame

cfg = load_config("configs/config.yaml", [
    "pipeline.detector=geco2",
    f"data.data_root={DATA_ROOT}",
    f"data.gt.global_file={TRAIN_GT_FILE}",
    f"project.work_dir={WORK_DIR}",
])

train_ids, val_ids = split_train_val(list_video_ids(TRAIN_GT_FILE), tuple(HOLDOUT_CATEGORIES))
check_videos = [train_ids[0], train_ids[-1], val_ids[0]]
print("Kiem tra toa do tren cac video:", check_videos)

# Baseline (chua finetune) detector -- chi de ve len anh minh hoa, KHONG dung de train.
detector = GeCo2Detector(cfg)

viz_dir = os.path.join(WORK_DIR, "viz", "geco2_finetune_gt_check")
os.makedirs(viz_dir, exist_ok=True)

for video_id in check_videos:
    gt = load_gt(TRAIN_GT_FILE, video_id)
    video_path = _locate_video(cfg, video_id)
    present_frames = sorted(gt.keys())
    check_frames = [present_frames[len(present_frames) // 2]] if present_frames else []
    # them 1 frame absent neu co (frame 0 thuong an toan la absent voi hau het video)
    if 0 not in gt:
        check_frames.append(0)

    for frame_idx in check_frames:
        frame_bgr = read_frame(video_path, frame_idx)
        gt_box = gt.get(frame_idx)
        padded, box_canvas, scale = convert_gt_box_to_canvas(frame_bgr, gt_box, image_size=cfg.stage123_geco2.image_size)

        # padded la tensor ImageNet-normalized -- undo de ve duoc
        vis = padded.permute(1, 2, 0).numpy()
        vis = (vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])) * 255.0
        vis = np.clip(vis, 0, 255).astype(np.uint8)[:, :, ::-1].copy()  # RGB -> BGR

        label = "PRESENT" if gt_box is not None else "ABSENT (no GT box)"
        if box_canvas is not None:
            x1, y1, x2, y2 = [int(round(c)) for c in box_canvas]
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)  # xanh la = GT (chuyen doi qua ham dung chung)

        out_path = os.path.join(viz_dir, f"{video_id}_frame{frame_idx}_{label.split()[0].lower()}.jpg")
        cv2.imwrite(out_path, vis)
        print(f"{video_id} frame {frame_idx} [{label}] -> {out_path}")

print("\n>>> MO CAC ANH TREN VA XAC NHAN BANG MAT: box xanh la phai nam DUNG tren vat the that")
print(">>> (voi frame PRESENT). KHONG chuyen sang muc 12 neu box bi lech/sai vi tri.")

## 12. Dry-run smoke test (~1-2 phut) -- chay TRUOC khi commit thoi gian thue GPU cho full training

Chi chay vai buoc, assert loss huu han (finite), KHONG luu checkpoint. Cac loi trong
`geco2_train_wrapper.py` (VD sai shape tensor, thieu aux head, ...) chi co the lo ra tren GPU
that -- khong co cach nao kiem tra truoc tren may local (khong co CUDA ops, khong co checkpoint).

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
python -m scripts.train_geco2_aeroeyes \
    --config configs/config.yaml \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$TRAIN_GT_FILE" \
    --set project.work_dir="$WORK_DIR" \
    --base-checkpoint "$BASE_CHECKPOINT" \
    --out-checkpoint "$OUT_CHECKPOINT" \
    --dry-run --dry-run-steps 2

## 13. Chay full training

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
python -m scripts.train_geco2_aeroeyes \
    --config configs/config.yaml \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$TRAIN_GT_FILE" \
    --set project.work_dir="$WORK_DIR" \
    --holdout-categories $HOLDOUT_CATEGORIES \
    --epochs $EPOCHS --steps-per-epoch $STEPS_PER_EPOCH --batch-size $BATCH_SIZE \
    --lr $LR --weight-decay $WEIGHT_DECAY --max-grad-norm $MAX_GRAD_NORM \
    --aux-weight $AUX_WEIGHT --p-present $P_PRESENT \
    --ref-downscale-lo $REF_DOWNSCALE_LO --ref-downscale-hi $REF_DOWNSCALE_HI \
    --base-checkpoint "$BASE_CHECKPOINT" --out-checkpoint "$OUT_CHECKPOINT" \
    --early-stop-patience $EARLY_STOP_PATIENCE --seed 42

## 13b. Calibrate `score_threshold_abs` cho checkpoint DA FINETUNE (chay TRUOC muc 14)

**Vi sao can lam lai buoc nay**: `class_embed`/`class_embed_aux` (dau ra sinh diem `box_v`/
centerness) nam trong nhom bi finetune -- thang diem so sau khi finetune KHONG dam bao con giong
truoc finetune. Mot gia tri `score_threshold_abs` da calibrate tren checkpoint GOC (vd 0.5) khong
co co so gi de dung tiep cho checkpoint MOI -- day chinh la kieu "doan tham so khong co can cu" ma
ca ke hoach nay ton tai de loai bo (giong het van de `expected_object_px`).

**Vi sao calibrate tren 4 video VAL (`Lifering_0/1`, `Person1_0/1`) chu KHONG PHAI 6 video TEST**:
neu calibrate thang tren 6 video test roi lai dung chinh 6 video do de danh gia ST-IoU o muc 14 ->
ro ri du lieu danh gia (data leakage), lam con so ST-IoU cuoi cung khong con khach quan. 4 video val
tuy khong nhan gradient (da bi holdout khoi training that) nhung van cung domain voi 10 video
training -- dung de calibrate la hop ly va tach bach ro rang voi buoc danh gia cuoi cung.

Doc output: neu "SEPARABLE" xuat hien voi ca 4 video, ghi lai gia tri `floor` de dung o muc 14. Neu
phan lon "NOT cleanly separable", giu `score_threshold_abs=0.0` (tat, hanh vi cu) va bao lai --
nghia la domain-randomization/absent-frame training chua tach biet duoc score present/absent ro
nhu ky vong.

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
for sid in Lifering_0 Lifering_1 Person1_0 Person1_1; do
    python -m scripts.check_geco2_score_separation \
        --config configs/config.yaml \
        --set stage123_geco2.weights_path="$OUT_CHECKPOINT" \
        --set stage123_geco2.use_shape_token=false \
        --set stage123_geco2.scale_calibration.enabled=false \
        --set stage123_geco2.ref_downscale_factor=1.0 \
        --set data.data_root="$DATA_ROOT" \
        --set data.gt.global_file="$TRAIN_GT_FILE" \
        --sample "$sid"
done

In [ ]:
# <-- CHINH theo gia tri "floor" in ra o cell tren (trung binh/thap nhat trong 4 video, tuy muc do
# dong nhat). 0.0 = giu hanh vi cu (khong loc absent-frame o inference) neu 4 video tren KHONG
# tach biet duoc ro rang.
SCORE_THRESHOLD_ABS = 0.0

import os
os.environ["SCORE_THRESHOLD_ABS"] = str(SCORE_THRESHOLD_ABS)
print("SCORE_THRESHOLD_ABS se dung o muc 14:", SCORE_THRESHOLD_ABS)

## 14. Danh gia checkpoint da finetune tren 6 video test held-out

Tai su dung `aero_eyes.stages.run_all` / `aero_eyes.evaluate` **HOAN TOAN KHONG SUA** -- diem
khac biet duy nhat la 5 `--set` duoi day (checkpoint moi + tat shape token + tat scale_calibration
+ dat ref_downscale_factor=1.0 + score_threshold_abs vua calibrate o muc 13b).
`ref_downscale_factor=1.0` la CO CHU DICH: muc dich cua domain-randomization luc training la de
model KHONG CAN doan truoc do net nao -- dat gia tri khac 1.0 se tai tao dung cai kieu doan-mu ma
ke hoach nay ton tai de loai bo.

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    --set pipeline.detector=geco2 \
    --set stage123_geco2.weights_path="$OUT_CHECKPOINT" \
    --set stage123_geco2.use_shape_token=false \
    --set stage123_geco2.scale_calibration.enabled=false \
    --set stage123_geco2.ref_downscale_factor=1.0 \
    --set stage123_geco2.score_threshold_abs="$SCORE_THRESHOLD_ABS" \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$TEST_GT_FILE" \
    --set project.work_dir="$WORK_DIR/finetuned_eval"

python -m aero_eyes.evaluate \
    --pred "$WORK_DIR/finetuned_eval/submission_all.json" \
    --gt "$TEST_GT_FILE" \
    --config configs/config.yaml

### (Tuy chon) So sanh truc tiep voi baseline (checkpoint goc + heuristic tot nhat da biet)

Chay lai cung 6 video test bang checkpoint GOC (chua finetune) de so sanh ST-IoU truc tiep --
theo tieu chi chap nhan cua ke hoach: checkpoint finetune phai **ro rang vuot** heuristic-only
pipeline, VA quan trong khong kem, phai lam duoc dieu do **khong can uoc luong `expected_object_px`**
(bao ca 2 chi so, khong chi rieng ST-IoU) -- chinh cac `--set` duoi day theo config heuristic tot
nhat da tung dung cho 6 video nay truoc do trong project.

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    --set pipeline.detector=geco2 \
    --set stage123_geco2.weights_path="$BASE_CHECKPOINT" \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$TEST_GT_FILE" \
    --set project.work_dir="$WORK_DIR/baseline_eval"

python -m aero_eyes.evaluate \
    --pred "$WORK_DIR/baseline_eval/submission_all.json" \
    --gt "$TEST_GT_FILE" \
    --config configs/config.yaml

## 15. Dong goi ket qua de tai ve

vast.ai khong tu luu output nhu Kaggle's Output tab -- **tai file nay ve TRUOC khi dung/huy
instance**, qua file browser cua Jupyter (chuot phai -> Download) hoac
`scp -P <port> root@<host>:/workspace/aero_eyes_finetune_results.zip .` (lay host/port tu nut
Connect tren trang instance vast.ai).

In [ ]:
%%bash
set -e
cd /workspace
rm -f aero_eyes_finetune_results.zip
zip -q -r aero_eyes_finetune_results.zip \
    "$OUT_CHECKPOINT" \
    "$WORK_DIR/finetuned_eval/submission_all.json" \
    "$WORK_DIR/baseline_eval/submission_all.json" \
    "$WORK_DIR/viz/geco2_finetune_gt_check" \
    2>/dev/null || true
ls -lh aero_eyes_finetune_results.zip
echo "Da dong goi: /workspace/aero_eyes_finetune_results.zip (checkpoint + ca 2 ket qua eval + viz sanity-check)" 